In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

import gpjax as gpx

In [ ]:
cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
from admpe.utils.helper import pprint_helper
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results,
    load_modeling_result,
    provide_modeling_results_overview,
    load_excitation_result,
    load_testing_sequences,
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result, evaluate_model_prediction, evaluate_model_rollout,
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts

In [ ]:
results = list_modeling_results(
    system_name=None,
    model_type="neural_euler",
    status="completed",
    exp_name="noiseless_w_obj",
    excitation_objective=["dmpe_uniform", "dmpe_static_dynamics"],
)
print(len(results))
results

In [ ]:
provide_modeling_results_overview(results)

## basic model performance metrics:

#### prediction:

In [ ]:
result = load_modeling_result(results[0]["path"])
wrapped_model, wrapped_env = get_wrapped_model_and_env(result)
evaluate_model_prediction(wrapped_model, wrapped_env)

In [ ]:
visualize_model_prediction(20, wrapped_model, wrapped_env)

In [ ]:
excitation_result = load_corresponding_excitation_result(result)
excitation_result["estimator"].visualize()
plt.show()

plt.plot(excitation_result["observations"])
plt.show()

#### rollouts

In [ ]:
modeling_result = load_modeling_result(results[0]["path"])
load_corresponding_excitation_result(modeling_result)["metadata"]

In [ ]:
signal_type = "ocp"

modeling_result = load_modeling_result(results[0]["path"])
excitation_result = load_corresponding_excitation_result(modeling_result)
testing_data = load_testing_sequences(modeling_result["metadata"]["system_name"])

wrapped_model, wrapped_env = get_wrapped_model_and_env(modeling_result)
evaluate_model_rollout(
    wrapped_model,
    observations=testing_data[signal_type]["observations"],
    actions=testing_data[signal_type]["actions"],
    horizons=np.arange(50, testing_data[signal_type]["observations"].shape[1], 50),
)

visualize_model_rollouts(
    wrapped_model,
    observations=jnp.concatenate([
        testing_data[signal_type]["observations"][0:5, 0:],
    ], axis=0),
    actions=jnp.concatenate([
        testing_data[signal_type]["actions"][0:5, 0:],
    ], axis=0),
)

In [ ]:
pprint_helper(testing_data)

- are the inaccuracies, esp. for fluid_tank and cart_pole, due to insufficient training or are the models really that bad at predicting stepping behavior?

In [ ]:
# results = list_modeling_results(
#     system_name="fluid_tank",
#     model_type="neural_euler",
#     status="completed",
#     exp_name="run1"
# )

signal_type = "ocp"

modeling_result = load_modeling_result(results[1]["path"])
excitation_result = load_corresponding_excitation_result(modeling_result)
testing_data = load_testing_sequences(modeling_result["metadata"]["system_name"])

wrapped_model, wrapped_env = get_wrapped_model_and_env(modeling_result)
evaluate_model_rollout(
    wrapped_model,
    observations=testing_data[signal_type]["observations"],
    actions=testing_data[signal_type]["actions"],
    horizons=np.arange(50, testing_data[signal_type]["observations"].shape[1], 50),
)

visualize_model_rollouts(
    wrapped_model,
    observations=jnp.concatenate([
        testing_data[signal_type]["observations"][0:5, 0:],
    ], axis=0),
    actions=jnp.concatenate([
        testing_data[signal_type]["actions"][0:5, 0:],
    ], axis=0),
)

In [ ]:
evaluate_model_rollout(
    wrapped_model,
    observations=testing_data[signal_type]["observations"],
    actions=testing_data[signal_type]["actions"],
    horizons=np.arange(50, testing_data[signal_type]["observations"].shape[1], 50),
)